In [65]:
import warnings
warnings.filterwarnings('ignore')

In [55]:
# !pip install crewai crewai_tools langchain_community

In [66]:
import openai
from dotenv import load_dotenv
env_path = r'/put your path here/.env'
load_dotenv(env_path)

True

In [67]:
from crewai import Agent, Task, Crew

In [68]:
customer_assist_agent = Agent(
    role="Customer Care Specialist",
    goal="Deliver the most courteous and effective "
         "customer support on your team.",
    backstory=(
        "You are a valued team member at Flight Travel Agency  (https://www.skyscanner.net/) "
        "and currently assigned to assist {customer}, a crucial client "
        "for the company."
        "Your mission is to ensure the highest quality of support!"
        "Provide detailed and thorough responses, "
        "avoiding any assumptions."
    ),
    allow_delegation=False,
    verbose=True
)

In [70]:
quality_check_agent = Agent(
    role="Support Quality Analyst",
    goal="Be recognized for ensuring the highest "
         "standards of support quality in your team.",
    backstory=(
        "You are a key contributor at Flight Travel Agency  (https://www.skyscanner.net/), "
        "working alongside your team to review a support request from {customer}. "
        "Your responsibility is to ensure that the Customer Care Specialist "
        "delivers top-notch service.\n"
        "It’s your duty to confirm that all responses are thorough, "
        "accurate, and free of assumptions."
    ),
    verbose=True
)

In [75]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

In [76]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://www.skyscanner.net/",config={}
)

In [77]:
customer_query_task = Task(
    description=(
        "{customer} has submitted a critical request:\n"
        "{inquiry}\n\n"
        "{person} from {customer} initiated the inquiry. "
        "Leverage all available resources and knowledge to "
        "deliver outstanding support."
        "Your goal is to provide a thorough and accurate resolution "
        "to the customer's question."
    ),
    expected_output=(
        "A comprehensive and well-researched response to the customer's inquiry "
        "that addresses every aspect of their concern.\n"
        "The response must reference all the tools and information sources used, "
        "including external materials or solutions. "
        "Ensure that the answer is exhaustive, leaves no room for follow-up questions, "
        "and maintains a professional, approachable tone."
    ),
    tools=[docs_scrape_tool],
    agent=customer_assist_agent,
)

In [78]:
response_review_task = Task(
    description=(
        "Evaluate the response prepared by the Customer Care Specialist for {customer}'s inquiry. "
        "Ensure the reply meets the highest standards of accuracy, completeness, and clarity "
        "expected in customer service.\n"
        "Confirm that every aspect of the customer's question has been addressed "
        "in a friendly and approachable manner.\n"
        "Verify the inclusion of appropriate references and sources used "
        "to gather the information, ensuring the response is well-supported and "
        "leaves no loose ends."
    ),
    expected_output=(
        "A polished, detailed, and ready-to-send response that completely addresses "
        "the customer's inquiry.\n"
        "The response should reflect all necessary feedback and adjustments while "
        "maintaining a relaxed yet professional tone that aligns with our cool and casual company culture."
    ),
    agent=quality_check_agent,
    llm=""
)

In [79]:
crewCSMAS = Crew(
  agents=[customer_assist_agent, quality_check_agent],
  tasks=[customer_query_task, response_review_task],
  verbose=True,
  memory=True
)

In [ ]:
inputs = {
    "customer": "Travel Inquiry",
    "person": "Deep Kulshrestha",
    "inquiry": "I need help with booking flight via UAE to New Delhi from Amsterdam,"
               " can you find the flight with minimum flying hours and cheapest option?"
}
result = crewCSMAS.kickoff(inputs=inputs)